# Construcción del Modelo de Regresión: Predicción de Alquileres

**Objetivo:** Desarrollar y evaluar un modelo de Machine Learning capaz de predecir el precio de alquiler de bienes inmuebles en Ecuador basándose en características físicas y de ubicación.

**Variables de entrada (Features):**
* Provincia
* Lugar
* Número de dormitorios
* Número de baños
* Área
* Número de garajes

**Variable objetivo (Target):** Precio

In [ ]:
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


ruta_datos = '../data/real_state_ecuador_limpio.csv' 
df = pd.read_csv(ruta_datos)

columnas_requeridas = ['Provincia', 'Lugar', 'Num. dormitorios', 'Num. banos', 'Area', 'Num. garages', 'Precio']
df_modelo = df[columnas_requeridas].copy()

print(f"Total de registros para entrenar: {df_modelo.shape[0]}")
display(df_modelo.head())

### 1. Preparación de Datos y División (Train/Test Split)
Separamos la variable objetivo (`Precio`) de las variables predictoras (features). Luego, dividimos el conjunto de datos asignando un 80% para entrenar el modelo y un 20% para probar su precisión con datos "nuevos".

Adicionalmente, configuramos un `ColumnTransformer` para:
* **Variables Numéricas:** Escalar los valores para que tengan un peso equitativo en el modelo (`StandardScaler`).
* **Variables Categóricas:** Convertir las zonas geográficas (`Provincia`, `Lugar`) en matrices binarias para que el algoritmo pueda procesarlas (`OneHotEncoder`).

In [ ]:
X = df_modelo.drop(columns=['Precio'])
y = df_modelo['Precio']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

cols_numericas = ['Num. dormitorios', 'Num. banos', 'Area', 'Num. garages']
cols_categoricas = ['Provincia', 'Lugar']

preprocesador = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), cols_numericas),
        ('cat', OneHotEncoder(handle_unknown='ignore'), cols_categoricas)
    ])

print("Datos divididos y preprocesador configurado correctamente.")
print(f"Propiedades para entrenar (Train): {X_train.shape[0]}")
print(f"Propiedades para probar (Test): {X_test.shape[0]}")

### 2. Construcción y Justificación del Modelo

**Modelo Seleccionado:** `RandomForestRegressor` (Bosque Aleatorio de Regresión)

**Justificación de la elección:**
1. **Manejo de no linealidad:** El mercado inmobiliario no es estrictamente lineal (ej. el precio no siempre sube proporcionalmente al área en todos los barrios por igual). Random Forest captura estas relaciones complejas.
2. **Robustez:** Es altamente resistente a la multicolinealidad y funciona de manera excelente con datos mixtos (variables numéricas continuas y categóricas codificadas con *One-Hot Encoding*).
3. **Poca sensibilidad a escalas:** Aunque estandarizamos los datos como buena práctica, este modelo basado en árboles de decisión no asume distribuciones estrictas de los datos, lo que lo hace ideal para datos reales del mercado ecuatoriano.

In [ ]:
pipeline_rf = Pipeline(steps=[
    ('preprocesamiento', preprocesador),
    ('modelo', RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1))
])

pipeline_rf.fit(X_train, y_train)

y_pred = pipeline_rf.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("--- EVALUACIÓN DEL DESEMPEÑO DEL MODELO ---")
print(f"Error Absoluto Medio (MAE): ${mae:.2f}")
print(f"Raíz del Error Cuadrático Medio (RMSE): ${rmse:.2f}")
print(f"Coeficiente de Determinación (R²): {r2:.4f}")

comparacion = pd.DataFrame({'Precio Real': y_test.values[:5], 'Predicción del Modelo': y_pred[:5]})
print("\n--- MUESTRA: REALIDAD VS PREDICCIÓN ---")
display(comparacion.round(2))

### 3. Análisis de Resultados y Limitaciones del Modelo

Al evaluar las métricas, observamos que el modelo base presenta un amplio margen de error y un poder predictivo limitado. Este comportamiento es **estadísticamente esperado y coherente** debido a dos factores críticos en el dataset:

1. **Volumen de Muestra Reducido (Subajuste):** Tras el proceso de limpieza, remoción de nulos y normalización de texto, el algoritmo fue entrenado con apenas **265 registros** y evaluado con **67**. Los modelos no lineales como *Random Forest* requieren un volumen sustancialmente mayor (miles de observaciones) para identificar patrones complejos de ramificación y generalizar correctamente hacia datos nuevos.
2. **Ausencia de Variables Determinantes:** El precio de los alquileres es altamente multifactorial. Las características proporcionadas (Área, Habitaciones, Baños, Ubicación y Garajes) representan solo una visión parcial del valor real del inmueble. El modelo no logra explicar gran parte del precio debido a la falta de variables clave en el dataset, tales como: antigüedad de la propiedad, si se renta amoblada o vacía, piso en el que se ubica, amenidades del edificio (piscina, gimnasio) y proximidad a vías principales.

**Conclusión:**
El *pipeline* de preprocesamiento (escalado y codificación *One-Hot*) y el algoritmo seleccionado son la arquitectura correcta para este tipo de problema. Sin embargo, para alcanzar un nivel de precisión apto para un entorno de producción, el siguiente paso lógico no es afinar hiperparámetros, sino ejecutar una fase de **enriquecimiento de datos (*Data Enrichment*)**, aumentando tanto el volumen de registros como la cantidad de variables (features) recopiladas por propiedad.

In [ ]:
joblib.dump(pipeline_rf, '../models/modelo_rf_alquileres.pkl')